In [2]:
!pip install rdkit-pypi

   ---------------------------------------- 0.0/20.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/20.5 MB 4.8 MB/s eta 0:00:05
   ------ --------------------------------- 3.1/20.5 MB 10.8 MB/s eta 0:00:02
   --------- ------------------------------ 5.0/20.5 MB 9.4 MB/s eta 0:00:02
   ---------- ----------------------------- 5.2/20.5 MB 9.4 MB/s eta 0:00:02
   ---------- ----------------------------- 5.2/20.5 MB 9.4 MB/s eta 0:00:02
   ------------- -------------------------- 6.8/20.5 MB 5.7 MB/s eta 0:00:03
   ----------------- ---------------------- 8.9/20.5 MB 6.4 MB/s eta 0:00:02
   ------------------- -------------------- 10.0/20.5 MB 6.2 MB/s eta 0:00:02
   -------------------- ------------------- 10.5/20.5 MB 6.0 MB/s eta 0:00:02
   -------------------- ------------------- 10.5/20.5 MB 6.0 MB/s eta 0:00:02
   -------------------- ------------------- 10.5/20.5 MB 6.0 MB/s eta 0:00:02
   -----

In [4]:
# ============================================================
# 🧠 ADVANCED AI DRUG DISCOVERY GRADIO APP
# ============================================================

# Install Libraries
# pip install gradio rdkit-pypi pandas numpy
# pip install scikit-learn xgboost matplotlib seaborn joblib

# ============================================================
# 📚 IMPORT LIBRARIES
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

# ============================================================
# 📁 SAMPLE DATASET
# ============================================================

data = {
    "smiles": [
        "CCO",
        "CCN",
        "CCC",
        "CCCl",
        "CCBr",
        "CC=O",
        "CC(C)O",
        "CC(C)N",
        "CCCC",
        "CCCN"
    ],

    "activity": [
        1, 1, 0, 0, 0,
        1, 1, 1, 0, 1
    ]
}

df = pd.DataFrame(data)

# ============================================================
# 🧪 FEATURE ENGINEERING
# ============================================================

def calculate_descriptors(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [0] * 8

    return [

        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
        Descriptors.TPSA(mol),
        Descriptors.HeavyAtomCount(mol)
    ]


# ============================================================
# 📊 CREATE FEATURES
# ============================================================

features = df["smiles"].apply(calculate_descriptors)

X = pd.DataFrame(
    features.tolist(),
    columns=[
        "MolWt",
        "LogP",
        "HDonors",
        "HAcceptors",
        "RotatableBonds",
        "RingCount",
        "TPSA",
        "HeavyAtoms"
    ]
)

y = df["activity"]

# ============================================================
# ✂️ TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================================
# 🤖 TRAIN MODEL
# ============================================================

model = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

# ============================================================
# 📈 MODEL EVALUATION
# ============================================================

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)

print(f"\nModel Accuracy: {accuracy:.2f}")

# ============================================================
# 💾 SAVE MODEL
# ============================================================

joblib.dump(model, "drug_discovery_model.pkl")

# ============================================================
# 🔮 PREDICTION FUNCTION
# ============================================================

def predict_drug(smiles):

    try:

        # Generate descriptors
        feat = calculate_descriptors(smiles)

        feat_df = pd.DataFrame(
            [feat],
            columns=X.columns
        )

        # Prediction
        pred = model.predict(feat_df)[0]

        prob = model.predict_proba(feat_df)[0][1]

        # Result
        if pred == 1:
            result = "🟢 Active Drug Candidate"
        else:
            result = "🔴 Inactive Molecule"

        # Feature Importance Plot
        importance = model.feature_importances_

        fi = pd.DataFrame({

            "Feature": X.columns,

            "Importance": importance
        })

        fi = fi.sort_values(
            by="Importance",
            ascending=False
        )

        fig, ax = plt.subplots(figsize=(8, 5))

        ax.barh(
            fi["Feature"],
            fi["Importance"]
        )

        ax.invert_yaxis()

        ax.set_title("Feature Importance")

        return (

            result,

            f"{prob:.2f}",

            fi,

            fig
        )

    except Exception as e:

        return (

            "Error",

            str(e),

            pd.DataFrame(),

            None
        )

# ============================================================
# 🎨 GRADIO UI
# ============================================================

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown(
        """
        # 🧠 AI Drug Discovery System
        
        Predict whether a molecule is:
        - Active Drug Candidate
        - Inactive Molecule
        
        Built using:
        - RDKit
        - XGBoost
        - Gradio
        """
    )

    with gr.Row():

        smiles_input = gr.Textbox(
            label="Enter SMILES String",
            placeholder="Example: CCO"
        )

    predict_btn = gr.Button(
        "🔍 Predict Molecule"
    )

    result_output = gr.Textbox(
        label="Prediction Result"
    )

    prob_output = gr.Textbox(
        label="Probability Score"
    )

    importance_output = gr.Dataframe(
        label="Feature Importance"
    )

    plot_output = gr.Plot(
        label="Feature Importance Plot"
    )

    predict_btn.click(

        fn=predict_drug,

        inputs=smiles_input,

        outputs=[
            result_output,
            prob_output,
            importance_output,
            plot_output
        ]
    )

# ============================================================
# 🚀 LAUNCH APP
# ============================================================

app.launch()


Model Accuracy: 0.50


C:\Users\Sagar\AppData\Local\Temp\ipykernel_12192\3365483357.py:218: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
